In [ ]:
import networkx as nx
import numpy as np
import functools


In [ ]:
@functools.lru_cache(maxsize=None)  # 没有缓存限制
def included_node_shortest_path(graph: nx.Graph)-> dict['source_node' : '{target_node:[node, node]}']:
    '''
    计算图中所有节点对之间的最短路径及所经过的节点（包含源节点本身），并缓存结果。
    参数:
        graph
    返回:
        dict: 格式为 {source_node: {target_node: [node, node, ...]}} 的最短路径字典。
        '''
    
    return dict(nx.all_pairs_shortest_path(graph))

@functools.lru_cache(maxsize=None)  # 没有缓存限制
def nodes_degree(graph) -> dict['Node' : 'degree']:
    
    return dict(graph.degree())

@functools.lru_cache(maxsize=None)  # 没有缓存限制
def scater_value_of_node(graph:nx.Graph, belta)->dict['node' : 'value']:
    degree = nodes_degree(graph)
    
    return {node : value ** belta for node, value in degree.items()}

In [ ]:
import math
def radiation_centrality_from_i(graph:nx.Graph, i:'soude', gama, alph, belta)->float:
    '''
    计算节点i的辐射能力，返回值为float
    '''
    
    degree = nodes_degree(graph)
    all_shortest_path = included_node_shortest_path(graph)

    # 节点i最短路径中所经过的节点dict{target node : list[node in path]}
    through_node_derive_i = all_shortest_path[i]

    # 计算源节点i到目标节点的最短距离的大小
    shortest_distances = {}
    for node, path in through_node_derive_i.items():
        shortest_distances[node] = len(path) - 1
    
   # 获取每个节点的散射力
    scater_value = scater_value_of_node(graph, belta)

    # 初始化衰减后的值
    after_decay = 0

    # 遍历所有目标节点 j
    for j, path in through_node_derive_i.items():
        if j != i:  # 排除源节点自身
            scatter = 1

            # 计算路径中经过的节点的散射力
            for through_node in path:
                if through_node != i:  # 排除源节点自身
                    scatter *= scater_value[through_node]

            # 计算衰减后的值并累加
            decay = scatter / math.exp(alph * shortest_distances[j])
            after_decay += decay

    # 返回最终的辐射能力值
    return (degree[i] ** gama) * after_decay

In [ ]:
def radiation_centrality(graph:nx.Graph, gama, alph, belta)->dict['node', 'value']:
    radiation_capability = {}
    for node in graph.nodes():
        value = radiation_centrality_from_i(graph, node, gama, alph, belta)
        radiation_capability[node] = value

    return radiation_capability